The agent should:

- Read incoming customer emails
- Classify them by urgency and topic
- Search relevant documentation to answer questions
- Draft appropriate responses
- Escalate complex issues to human agents
- Schedule follow-ups when needed

Example scenarios to handle:

1. Simple product question: "How do I reset my password?"
2. Bug report: "The export feature crashes when I select PDF format"
3. Urgent billing issue: "I was charged twice for my subscription!"
4. Feature request: "Can you add dark mode to the mobile app?"
5. Complex technical issue: "Our API integration fails intermittently with 504 errors"

In [1]:
from typing import TypedDict,Literal

#define the sturcture for email classification

class EmailClassification(TypedDict):
    intent: Literal["question","bug","billing","feature","complex"]
    urgency: Literal["low","medium","high","critical"]
    topic:str
    summery:str


class EmailAgentState(TypedDict):
    # Raw email data
    email_content: str
    sender_email: str
    email_id:str

    #classification result
    classification : EmailClassification|None

    # Raw search/API results
    search_results :list[str] | None
    customer_history: dict | None

    #Generated content
    draft_response: str | None
    messages:list[dict] | None

In [19]:
from langgraph.graph import StateGraph,START,END
from langgraph.types import interrupt, Command,RetryPolicy
from typing import Literal
from langchain_groq import ChatGroq
from langchain.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.2)

In [20]:
# make our email reading node

def read_email(state:EmailAgentState) -> dict:
    """Extract and parse email content"""
    return {"messages":[HumanMessage(content=f"Processing email : {state['email_content']}")]}

In [21]:
# define classification_intent node

def classify_intent(state:EmailAgentState)->Command[Literal["search_documentation", "human_review", "draft_response", "bug_tracking"]]:
    """ use llm to classify the email intent and urgency, then route accordingly"""
    structured_llm=llm.with_structured_output(EmailClassification)

    #Format the prompt on demand, not stored in state

    classification_prompt =f"""Analyze the following email and classify it

    Email: {state['email_content']}
    From : {state['sender_email']}
    
    Provide classification including intent,ugency,topic and summery.
    """

    #get structure response directly as dict
    classification = structured_llm.invoke(classification_prompt)

    #Determine routing based on classification
    if classification["intent"] == "billing" or classification['urgency']=='critical':
        goto = "human_review"

    elif classification["intent"]=="bug":
        goto = "bug_tracking"
    elif classification["intent"] in ["question","feature"]:
        goto = "search_documentation"
    else:
        goto = "draft_response"

    #Store classification as single dict in state

    return Command(
        update={"classification": classification},
        goto=goto
    )




In [22]:
def search_documentation(state:EmailAgentState)->Command[Literal["draft_response"]]:
    """Search Knowledge base for relevant information"""
    #Build serach query from classification
    classification = state['classification',{}]
    query = f"{classification.get('intent',{})}{classification.get('topic',{})}"


    try: 
        #Implement a retry policy for search logic here
        # Store raw search results, not formatted text

        search_results = [
            "Resest password via Settings > Account > Reset Password",
            "Passward must be at least 12 characters",
            "Inlude uppercase,lowercase,number and special character"
        ]
    except SearchAPIError as e:
        # for recoverable search errors, store error and continue

        search_results = [f"Search temporarily unavailable: {str(e)}"]
    return Command(
        update={"search_results": search_results},
        goto="draft_response"
    )



In [23]:
def bug_tracking(state:EmailAgentState)->Command[Literal["draft_response"]]:
    """Create or update bug tracking ticket"""
    ticket_id="BUG-1234" #would be created via API call in real implementation
    return Command(
        update={
            "search_results":[f"Bug ticket {ticket_id} created"],
            "current_step":"bug_tracked"
        }
        ,goto="draft_response"
    )

In [24]:
def draft_response(state: EmailAgentState) -> Command[Literal["human_review", "send_reply"]]:
    """Generate response using context and route based on quality"""

    classification = state.get('classification', {})

    # Format context from raw state data on-demand
    context_sections = []

    if state.get('search_results'):
        # Format search results for the prompt
        formatted_docs = "\n".join([f"- {doc}" for doc in state['search_results']])
        context_sections.append(f"Relevant documentation:\n{formatted_docs}")

    if state.get('customer_history'):
        # Format customer data for the prompt
        context_sections.append(f"Customer tier: {state['customer_history'].get('tier', 'standard')}")

    # Build the prompt with formatted context
    draft_prompt = f"""
    Draft a response to this customer email:
    {state['email_content']}

    Email intent: {classification.get('intent', 'unknown')}
    Urgency level: {classification.get('urgency', 'medium')}

    {chr(10).join(context_sections)}

    Guidelines:
    - Be professional and helpful
    - Address their specific concern
    - Use the provided documentation when relevant
    """

    response = llm.invoke(draft_prompt)

    # Determine if human review needed based on urgency and intent
    needs_review = (
        classification.get('urgency') in ['high', 'critical'] or
        classification.get('intent') == 'complex'
    )

    # Route to appropriate next node
    goto = "human_review" if needs_review else "send_reply"

    return Command(
        update={"draft_response": response.content},  # Store only the raw response
        goto=goto
    )

def human_review(state: EmailAgentState) -> Command[Literal["send_reply", END]]:
    """Pause for human review using interrupt and route based on decision"""

    classification = state.get('classification', {})

    # interrupt() must come first - any code before it will re-run on resume
    human_decision = interrupt({
        "email_id": state.get('email_id',''),
        "original_email": state.get('email_content',''),
        "draft_response": state.get('draft_response',''),
        "urgency": classification.get('urgency'),
        "intent": classification.get('intent'),
        "action": "Please review and approve/edit this response"
    })

    # Now process the human's decision
    if human_decision.get("approved"):
        return Command(
            update={"draft_response": human_decision.get("edited_response", state.get('draft_response',''))},
            goto="send_reply"
        )
    else:
        # Rejection means human will handle directly
        return Command(update={}, goto=END)

def send_reply(state: EmailAgentState) -> dict:
    """Send the email response"""
    # Integrate with email service
    print(f"Sending reply: {state['draft_response'][:100]}...")
    return {}

In [25]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import RetryPolicy

# Create the graph
workflow = StateGraph(EmailAgentState)

# Add nodes with appropriate error handling
workflow.add_node("read_email", read_email)
workflow.add_node("classify_intent", classify_intent)

# Add retry policy for nodes that might have transient failures
workflow.add_node(
    "search_documentation",
    search_documentation,
    retry_policy=RetryPolicy(max_attempts=3)
)
workflow.add_node("bug_tracking", bug_tracking)
workflow.add_node("draft_response", draft_response)
workflow.add_node("human_review", human_review)
workflow.add_node("send_reply", send_reply)

# Add only the essential edges
workflow.add_edge(START, "read_email")
workflow.add_edge("read_email", "classify_intent")
workflow.add_edge("send_reply", END)

# Compile with checkpointer for persistence, in case run graph with Local_Server --> Please compile without checkpointer
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [26]:
# Test with an urgent billing issue
initial_state = {
    "email_content": "I was charged twice for my subscription! This is urgent!",
    "sender_email": "customer@example.com",
    "email_id": "email_123",
    "messages": []
}

# Run with a thread_id for persistence
config = {"configurable": {"thread_id": "customer_123"}}
result = app.invoke(initial_state, config)
# The graph will pause at human_review
print(f"human review interrupt:{result['__interrupt__']}")

# When ready, provide human input to resume
from langgraph.types import Command

human_response = Command(
    resume={
        "approved": True,
        "edited_response": "We sincerely apologize for the double charge. I've initiated an immediate refund..."
    }
)

# Resume execution
final_result = app.invoke(human_response, config)
print(f"Email sent successfully!")

human review interrupt:[Interrupt(value={'email_id': 'email_123', 'original_email': 'I was charged twice for my subscription! This is urgent!', 'draft_response': "Subject: Urgent: Duplicate Charge for Subscription\n\nDear [Customer's Name],\n\nI apologize for the inconvenience you've experienced with being charged twice for your subscription. I understand the urgency of this matter and am here to assist you in resolving it as quickly as possible.\n\nI have immediately looked into this issue and have created a bug ticket, BUG-1234, to track and address the problem. Our technical team will investigate this duplicate charge and work on preventing such errors in the future.\n\nIn the meantime, I want to assure you that we are taking steps to rectify the situation. I will personally ensure that the duplicate charge is refunded to you as soon as possible. You will receive a confirmation email once the refund has been processed.\n\nIf you have any further questions or concerns, please do not 